# Tutorial: DL-04 Diagnostics Demo

- Module: 07 Deep Learning Systems
- Topic: monitoring loss, gradients, activations, learning rates, and confusion matrices
- Estimated runtime: 3-5 minutes
- Prerequisites: Module 06 scratch MLP, Module 02 evaluation toolkit, and training pathologies from the Module 07 notes
- Expected outputs: stable and unstable training traces, gradient-flow plots, activation histograms, and a confusion matrix


## Learning goals

After running this notebook, you should be able to:

- reuse `shared/src/training_diagnostics.py` instead of writing notebook-local plotting code;
- distinguish a healthy run from depthwise gradient decay and exploding gradients;
- use activation histograms to diagnose hidden-unit saturation; and
- combine training diagnostics with a held-out confusion matrix for classifier evaluation.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from notebook path.")


REPO_ROOT = find_repo_root(Path.cwd())
for path in (REPO_ROOT, REPO_ROOT / "modules/06-neural-networks/src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from nn_from_scratch import ScratchMLP
from shared.src.evaluation import classification_metrics, train_valid_test_split
from shared.src.training_diagnostics import (
    activation_summary,
    append_gradient_snapshot,
    plot_activation_histograms,
    plot_confusion_diagnostics,
    plot_gradient_norms,
    plot_learning_rates,
    plot_loss_curves,
)

SEED = 11
np.random.seed(SEED)
SEED


11

## Step 1 - Build one dataset and one split policy

The point of the notebook is not dataset engineering. We keep the task small and use the shared train/validation/test split helper so the diagnostics stay comparable across runs.


In [2]:
X_raw, y = make_moons(n_samples=300, noise=0.18, random_state=SEED)
X = StandardScaler().fit_transform(X_raw)
split = train_valid_test_split(X, y, random_state=SEED, stratify=y)

{key: value.shape for key, value in split.items()}


{'X_train': (180, 2),
 'X_valid': (60, 2),
 'X_test': (60, 2),
 'y_train': (180,),
 'y_valid': (60,),
 'y_test': (60,)}

## Step 2 - Define a training loop that records diagnostics

The shared toolkit stays array-first. The training loop simply records loss, learning rate, and layerwise gradient norms, then keeps one activation snapshot from the final forward pass.


In [3]:
def scale_parameters(model: ScratchMLP, scale: float) -> None:
    for name, values in model.parameters.items():
        if name.startswith("W"):
            model.parameters[name] = values * scale


def build_activation_map(caches: list[dict[str, np.ndarray]]) -> dict[str, np.ndarray]:
    activation_map: dict[str, np.ndarray] = {}
    for index, cache in enumerate(caches[:-1], start=1):
        activation_name = str(cache["activation"][0])
        activation_map[f"layer_{index}_{activation_name}"] = cache["A"]
    return activation_map


def train_run(
    model: ScratchMLP,
    *,
    epochs: int,
    base_lr: float,
    decay: float,
    vanish_threshold: float = 1e-7,
    explode_threshold: float = 10.0,
) -> dict[str, object]:
    history = {"train_loss": [], "valid_loss": [], "learning_rate": []}
    gradient_history: dict[str, list[float]] = {}
    final_caches: list[dict[str, np.ndarray]] | None = None

    for epoch in range(1, epochs + 1):
        current_lr = base_lr * (decay ** (epoch - 1))
        y_pred, caches = model.forward(split["X_train"])
        train_loss = model.compute_loss(y_pred, split["y_train"])
        grads = model.backward(split["y_train"], caches)
        append_gradient_snapshot(
            gradient_history,
            grads,
            step=epoch,
            vanish_threshold=vanish_threshold,
            explode_threshold=explode_threshold,
        )
        model.update_parameters(grads, current_lr)
        history["train_loss"].append(train_loss)
        history["valid_loss"].append(model.loss_for_current_parameters(split["X_valid"], split["y_valid"]))
        history["learning_rate"].append(current_lr)
        final_caches = caches

    if final_caches is None:
        raise RuntimeError("Training loop did not record activations.")

    test_scores = model.predict_proba(split["X_test"]).reshape(-1)
    test_predictions = model.predict(split["X_test"])
    layer_norms = {
        name: values[-1]
        for name, values in gradient_history.items()
        if name.startswith("dW")
    }

    return {
        "history": history,
        "gradient_history": gradient_history,
        "activation_map": build_activation_map(final_caches),
        "activation_summary": activation_summary(build_activation_map(final_caches)),
        "layer_norms": layer_norms,
        "metrics": classification_metrics(split["y_test"], test_predictions, y_score=test_scores),
        "test_predictions": test_predictions,
    }


## Step 3 - Run three deliberately different training regimes

We use three models with the same dataset:

- a healthy ReLU baseline;
- a deep sigmoid stack with oversized weights, which tends to show strong depthwise gradient decay and hidden-unit saturation; and
- an aggressive ReLU stack with oversized weights and a large step size, which drives exploding gradients.


In [4]:
healthy_model = ScratchMLP([2, 16, 16, 1], ["relu", "relu", "sigmoid"], seed=SEED)
healthy_run = train_run(healthy_model, epochs=14, base_lr=0.12, decay=0.97, explode_threshold=5.0)

vanishing_model = ScratchMLP(
    [2, 32, 32, 32, 32, 32, 32, 32, 32, 1],
    ["sigmoid", "sigmoid", "sigmoid", "sigmoid", "sigmoid", "sigmoid", "sigmoid", "sigmoid", "sigmoid"],
    seed=SEED,
)
scale_parameters(vanishing_model, 8.0)
vanishing_run = train_run(vanishing_model, epochs=6, base_lr=0.05, decay=1.0, explode_threshold=15.0)

exploding_model = ScratchMLP([2, 32, 32, 32, 1], ["relu", "relu", "relu", "sigmoid"], seed=SEED)
scale_parameters(exploding_model, 8.0)
exploding_run = train_run(exploding_model, epochs=6, base_lr=0.18, decay=1.0, explode_threshold=15.0)

summary_rows = [
    {
        "run": "healthy_relu",
        "test_accuracy": round(float(healthy_run["metrics"]["accuracy"]), 3),
        "max_total_grad_norm": float(np.max(healthy_run["gradient_history"]["total_norm"])),
    },
    {
        "run": "deep_sigmoid_saturation",
        "test_accuracy": round(float(vanishing_run["metrics"]["accuracy"]), 3),
        "max_total_grad_norm": float(np.max(vanishing_run["gradient_history"]["total_norm"])),
        "input_to_output_grad_ratio": float(
            vanishing_run["layer_norms"]["dW9"] / max(vanishing_run["layer_norms"]["dW1"], 1e-12)
        ),
    },
    {
        "run": "aggressive_relu",
        "test_accuracy": round(float(exploding_run["metrics"]["accuracy"]), 3),
        "max_total_grad_norm": float(np.max(exploding_run["gradient_history"]["total_norm"])),
    },
]
summary_rows


[{'run': 'healthy_relu',
  'test_accuracy': 0.817,
  'max_total_grad_norm': 0.7562626941361077},
 {'run': 'deep_sigmoid_saturation',
  'test_accuracy': 0.467,
  'max_total_grad_norm': 1.9759275166725794,
  'input_to_output_grad_ratio': 10.116249350748152},
 {'run': 'aggressive_relu',
  'test_accuracy': 0.5,
  'max_total_grad_norm': 1.0991114650001295e+43}]

The shared tracker catches the exploding run immediately: its total gradient norm is astronomically larger than the healthy baseline. The deep sigmoid run is subtler. Its total norm is not catastrophic, but the earliest weight gradient is about an order of magnitude smaller than the output-side gradient, which is exactly the depthwise decay pattern we want to notice before optimization stalls.


## Step 4 - Inspect the healthy baseline

Start with the run that behaves well enough to learn the task. We want the training and validation losses to decrease together and the learning-rate schedule to be explicit rather than hidden in code.


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_loss_curves(
    healthy_run["history"]["train_loss"],
    healthy_run["history"]["valid_loss"],
    ax=axes[0],
    title="Healthy run: train vs validation loss",
)
plot_learning_rates(
    healthy_run["history"]["learning_rate"],
    ax=axes[1],
    title="Healthy run: learning-rate decay",
)
display(fig)
plt.close(fig)

healthy_run["metrics"]


<Figure size 1200x400 with 2 Axes>

{'accuracy': 0.8166666666666667,
 'precision': 0.8518518518518519,
 'recall': 0.7666666666666667,
 'f1': 0.8070175438596491,
 'roc_auc': 0.8944444444444444,
 'pr_auc': 0.8714024273099924,
 'brier_score': 0.14186809514446003}

## Step 5 - Compare gradient-flow pathologies

The plot below is the real reason to keep gradient diagnostics in a shared toolkit. The exploding run has norms that immediately leave the sane regime, while the deep sigmoid run shows the quieter failure mode: gradients exist, but the earliest layers receive much less signal than the output side.


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
plot_gradient_norms(
    vanishing_run["gradient_history"],
    ax=axes[0],
    title="Deep sigmoid stack: depthwise gradient decay",
)
plot_gradient_norms(
    exploding_run["gradient_history"],
    ax=axes[1],
    title="Aggressive ReLU stack: exploding gradients",
)
display(fig)
plt.close(fig)

{
    "deep_sigmoid_final_layer_norms": vanishing_run["layer_norms"],
    "aggressive_relu_peak_total_norm": float(np.max(exploding_run["gradient_history"]["total_norm"])),
}


<Figure size 1300x440 with 2 Axes>

{'deep_sigmoid_final_layer_norms': {'dW9': 0.3990934555660197,
  'dW8': 0.3726326346456242,
  'dW7': 0.24926988708078898,
  'dW6': 0.3079996779724953,
  'dW5': 0.40333586463035004,
  'dW4': 0.2733282030211122,
  'dW3': 0.25635869416261514,
  'dW2': 0.231603940125556,
  'dW1': 0.039450733342837635},
 'aggressive_relu_peak_total_norm': 1.0991114650001295e+43}

## Step 6 - Use activation histograms to confirm saturation

The deep sigmoid run is a good activation-diagnostics example because many hidden units have already moved close to `0` or `1`. That saturation makes derivatives small and helps explain the gradient decay we just saw.


In [7]:
hist_figure, _ = plot_activation_histograms(
    vanishing_run["activation_map"],
    title="Deep sigmoid activations after training",
)
display(hist_figure)
plt.close(hist_figure)

{name: values["saturation_fraction"] for name, values in vanishing_run["activation_summary"].items()}


<Figure size 1240x1520 with 8 Axes>

{'layer_1_sigmoid': 0.6642361111111111,
 'layer_2_sigmoid': 0.5522569444444444,
 'layer_3_sigmoid': 0.6427083333333333,
 'layer_4_sigmoid': 0.7072916666666667,
 'layer_5_sigmoid': 0.7090277777777778,
 'layer_6_sigmoid': 0.5256944444444445,
 'layer_7_sigmoid': 0.6689236111111111,
 'layer_8_sigmoid': 0.5782986111111111}

## Step 7 - Close the loop with held-out predictions

Training diagnostics tell us whether optimization is healthy, but we still need an evaluation view on held-out data. The shared confusion-matrix helper keeps that plot visually aligned with the earlier evaluation toolkit from Module 02.


In [8]:
fig, ax = plt.subplots(figsize=(4.8, 4.2))
plot_confusion_diagnostics(split["y_test"], healthy_run["test_predictions"], ax=ax)
display(fig)
plt.close(fig)

{
    "healthy_test_metrics": healthy_run["metrics"],
    "diagnostic_takeaway": "Use loss, gradient, activation, and confusion-matrix views together before changing architecture or optimizer settings.",
}


<Figure size 480x420 with 2 Axes>

{'healthy_test_metrics': {'accuracy': 0.8166666666666667,
  'precision': 0.8518518518518519,
  'recall': 0.7666666666666667,
  'f1': 0.8070175438596491,
  'roc_auc': 0.8944444444444444,
  'pr_auc': 0.8714024273099924,
  'brier_score': 0.14186809514446003},
 'diagnostic_takeaway': 'Use loss, gradient, activation, and confusion-matrix views together before changing architecture or optimizer settings.'}

## Exercises

- Lower the sigmoid-stack initialization scale from `8.0` to `2.0` and see how the activation histograms change.
- Add gradient clipping to the aggressive ReLU run and compare its peak total norm to the unclipped baseline.
- Replace the constant learning rate in the unstable runs with a decayed schedule and check whether the confusion matrix improves.
